# Working with Complex Data

So far every column has held a single value: a number, a string, a
timestamp. Spark columns can also hold **nested** values, which lets one row
carry a list of items, a record of related fields, or a whole JSON document.
Such data shows up constantly in practice, from API responses and event logs
to the output of `collect_list` that appeared in
{doc}`Grouping and Aggregating Data <02_grouping_and_aggregating_data>`.

| Type | Holds | Example |
| --- | --- | --- |
| `ARRAY<T>` | An ordered list of values of one type | `["Pearly Pies", "Orchard Oasis"]` |
| `STRUCT<a: T1, b: T2>` | A fixed set of named fields, like a row inside a row | `{city: "Tokyo", country: "Japan"}` |
| `MAP<K, V>` | Key-value pairs with keys of one type and values of another | `{"visa": 12, "amex": 3}` |
| `VARIANT` | Any semi-structured value, typically parsed JSON | `{"rating": 5, "tags": ["sweet"]}` |

This notebook shows how to build, query and flatten each of them.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

transactions = spark.read.table("samples.bakehouse.sales_transactions")
franchises = spark.read.table("samples.bakehouse.sales_franchises")
reviews = spark.read.table("samples.bakehouse.media_customer_reviews")

## Arrays

### Building Arrays

Arrays usually come from one of three places: a source that already
contains them, aggregation with `collect_list` or `collect_set`, or
splitting a string. Here is each customer's purchase history, collected into
arrays:

In [2]:
baskets = transactions.groupBy("customerID").agg(
    F.sort_array(F.collect_set("product")).alias("products"),
    F.collect_list("quantity").alias("quantities"),
)

baskets.printSchema()
baskets.show(3, truncate=60)

root
 |-- customerID: long (nullable = true)
 |-- products: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- quantities: array (nullable = false)
 |    |-- element: long (containsNull = false)



+----------+------------------------------------------------------------+-------------------------------------------------------+
|customerID|                                                    products|                                             quantities|
+----------+------------------------------------------------------------+-------------------------------------------------------+
|   2000000|[Austin Almond Biscotti, Outback Oatmeal, Pearly Pies, To...|                          [3, 2, 1, 2, 10, 5, 1, 7, 36]|
|   2000001|[Austin Almond Biscotti, Golden Gate Ginger, Outback Oatm...|                             [3, 4, 10, 14, 8, 1, 4, 7]|
|   2000002|[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasi...|[7, 7, 5, 10, 2, 2, 8, 4, 8, 1, 6, 10, 9, 10, 10, 1, 9]|
+----------+------------------------------------------------------------+-------------------------------------------------------+
only showing top 3 rows


### Querying Arrays

Array functions work on the whole array without unpacking it. Indexing with
`[i]` is zero-based, while `element_at` is one-based and accepts negative
indices that count from the end:

In [3]:
baskets.select(
    "customerID",
    F.size("products").alias("distinct_products"),
    F.array_contains("products", "Pearly Pies").alias("bought_pearly_pies"),
    F.col("products")[0].alias("first_alphabetically"),
    F.element_at("products", -1).alias("last_alphabetically"),
    F.array_max("quantities").alias("largest_quantity"),
).show(5, truncate=False)

+----------+-----------------+------------------+----------------------+-------------------+----------------+
|customerID|distinct_products|bought_pearly_pies|first_alphabetically  |last_alphabetically|largest_quantity|
+----------+-----------------+------------------+----------------------+-------------------+----------------+
|2000000   |4                |true              |Austin Almond Biscotti|Tokyo Tidbits      |36              |
|2000001   |4                |false             |Austin Almond Biscotti|Tokyo Tidbits      |14              |
|2000002   |5                |true              |Austin Almond Biscotti|Pearly Pies        |10              |
|2000003   |5                |true              |Austin Almond Biscotti|Pearly Pies        |10              |
|2000004   |6                |true              |Austin Almond Biscotti|Tokyo Tidbits      |16              |
+----------+-----------------+------------------+----------------------+-------------------+----------------+
only showi

Other useful functions include `array_distinct`, `array_union`,
`array_intersect`, `array_except`, `arrays_overlap`, `slice` and
`array_join`, which concatenates an array of strings into one string.

### Exploding Arrays into Rows

`explode` turns each element of an array into its own row, repeating the
other columns. It is the inverse of `collect_list`:

In [4]:
baskets.select("customerID", F.explode("products").alias("product")).show(8)

+----------+--------------------+
|customerID|             product|
+----------+--------------------+
|   2000076|Austin Almond Bis...|
|   2000076|  Golden Gate Ginger|
|   2000076|       Orchard Oasis|
|   2000076|     Outback Oatmeal|
|   2000076|         Pearly Pies|
|   2000076|       Tokyo Tidbits|
|   2000000|Austin Almond Bis...|
|   2000000|     Outback Oatmeal|
+----------+--------------------+
only showing top 8 rows


:::::{dropdown} Example: How Many Rows Does `explode` Produce?
:open:
:color: primary
:icon: pencil
:class-body: example-body

A DataFrame has three rows whose arrays hold 2, 0 and 3 elements. How many
rows come out of `explode`, and how many out of `explode_outer`?

**`explode`.** Each row turns into one row per element, and a row whose
array is empty or `NULL` disappears completely:

$$
2 + 0 + 3 = 5 \text{ rows}
$$

**`explode_outer`.** Same as `explode`, but an empty or `NULL` array yields
one row with a `NULL` element, so the input row is not lost:

$$
2 + 1 + 3 = 6 \text{ rows}
$$

Use `explode_outer` whenever rows without elements must survive, for example
before a count that should include zero.
:::::

Checking both counts in Spark:

In [5]:
orders = spark.createDataFrame(
    [("A", ["Pearly Pies", "Tokyo Tidbits"]), ("B", []), ("C", ["Orchard Oasis", "Pearly Pies", "Outback Oatmeal"])],
    schema="order_id STRING, items ARRAY<STRING>",
)

print("explode:", orders.select("order_id", F.explode("items")).count())
print("explode_outer:", orders.select("order_id", F.explode_outer("items")).count())

explode: 5


explode_outer: 6


`posexplode` also returns each element's position, which preserves the
original order:

In [6]:
orders.select("order_id", F.posexplode("items").alias("position", "item")).show()

+--------+--------+---------------+
|order_id|position|           item|
+--------+--------+---------------+
|       A|       0|    Pearly Pies|
|       A|       1|  Tokyo Tidbits|
|       C|       0|  Orchard Oasis|
|       C|       1|    Pearly Pies|
|       C|       2|Outback Oatmeal|
+--------+--------+---------------+



### Higher-Order Functions

Exploding, transforming and re-collecting an array is expensive, because it
multiplies rows and then needs a `groupBy`. **Higher-order functions** work
on the elements in place instead, taking a Python lambda that Spark turns
into an expression. It runs inside the engine, not in Python, so it is as
fast as any other built-in function.

| Function | Does |
| --- | --- |
| `F.transform(arr, f)` | Applies `f` to every element |
| `F.filter(arr, f)` | Keeps elements for which `f` is true |
| `F.exists(arr, f)` / `F.forall(arr, f)` | Whether `f` holds for some / all elements |
| `F.aggregate(arr, zero, merge)` | Folds the array into one value |

In [7]:
baskets.select(
    "customerID",
    "quantities",
    F.transform("quantities", lambda q: q * 3).alias("prices"),
    F.filter("quantities", lambda q: q >= 10).alias("bulk_quantities"),
    F.exists("quantities", lambda q: q >= 30).alias("has_large_order"),
    F.aggregate("quantities", F.lit(0).cast("bigint"), lambda acc, q: acc + q).alias("items_bought"),
).show(3, truncate=40)

+----------+----------------------------------------+----------------------------------------+---------------+---------------+------------+
|customerID|                              quantities|                                  prices|bulk_quantities|has_large_order|items_bought|
+----------+----------------------------------------+----------------------------------------+---------------+---------------+------------+
|   2000253|[8, 1, 1, 4, 3, 1, 8, 6, 3, 10, 6, 4,...|[24, 3, 3, 12, 9, 3, 24, 18, 9, 30, 1...|       [10, 10]|          false|          76|
|   2000226|              [36, 1, 1, 8, 5, 8, 10, 1]|          [108, 3, 3, 24, 15, 24, 30, 3]|       [36, 10]|           true|          70|
|   2000108|[40, 6, 5, 4, 8, 8, 2, 1, 3, 7, 9, 2,...|[120, 18, 15, 12, 24, 24, 6, 3, 9, 21...|           [40]|           true|         114|
+----------+----------------------------------------+----------------------------------------+---------------+---------------+------------+
only showing top 3 r

:::::{dropdown} Example: Folding an Array with `aggregate`
:open:
:color: primary
:icon: pencil
:class-body: example-body

Take customer 2000226 from the output above, whose `quantities` are
`[36, 1, 1, 8, 5, 8, 10, 1]`, and follow `aggregate` with a starting value
of 0 and the merge function `acc + q`.

**Start from the zero value.** The accumulator `acc` begins at 0.

**Merge each element in turn.** For every quantity `q`, the new accumulator
is `acc + q`:

```text
acc:  0 -> 36 -> 37 -> 38 -> 46 -> 51 -> 59 -> 69 -> 70
```

After the last element, `acc` holds the sum of all quantities:

$$
36 + 1 + 1 + 8 + 5 + 8 + 10 + 1 = 70
$$

**Read the result.** The `items_bought` column for that customer shows the
same value, 70.

The zero value is cast to `bigint` to match the element type of
`quantities`. If the types differ, Spark refuses the expression instead of
silently converting the values.
:::::

### Splitting Text: Counting Words in Reviews

Splitting text into an array of words and exploding it is the classic word
count, and it works the same on one review as on a billion. Which words come
up most often in the bakehouse's online reviews?

In [8]:
stop_words = ["the", "and", "a", "to", "of", "i", "was", "is", "it", "with", "in", "for", "that", "this", "my", "their", "on", "as", "you", "an", "were", "but", "are", "be", "at", "me", "from", "had", "they", "s", "t", "so", "all", "just", "or", "its", "have", "by", "not", "we"]

words = (
    reviews
    .select(F.split(F.lower("review"), r"[^a-z]+").alias("words"))
    .select(F.explode("words").alias("word"))
    .filter((F.length("word") > 0) & ~F.col("word").isin(stop_words))
)

words.groupBy("word").count().orderBy(F.desc("count"), "word").show(10)

+---------+-----+
|     word|count|
+---------+-----+
|bakehouse|  442|
|   almond|  309|
|  oatmeal|  305|
|   cookie|  276|
| biscotti|  273|
|  cookies|  272|
|   austin|  238|
|  outback|  235|
|   ginger|  213|
|  orchard|  188|
+---------+-----+
only showing top 10 rows


Unsurprisingly, reviewers mostly talk about the bakery and its products.
Notice that `cookie` and `cookies` are counted separately: reducing words to
a common stem is a job for a natural language processing library, not for
`split`.

## Structs

A struct groups several named fields into a single column, like a small row
nested inside the row. Build one with `F.struct`:

In [9]:
stores = franchises.select(
    "franchiseID",
    "name",
    F.struct("city", "district", "country").alias("location"),
    F.struct("latitude", "longitude").alias("coordinates"),
)

stores.printSchema()
stores.show(3, truncate=False)

root
 |-- franchiseID: long (nullable = true)
 |-- name: string (nullable = true)
 |-- location: struct (nullable = false)
 |    |-- city: string (nullable = true)
 |    |-- district: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- coordinates: struct (nullable = false)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)



+-----------+----------------+--------------------------------+--------------------+
|franchiseID|name            |location                        |coordinates         |
+-----------+----------------+--------------------------------+--------------------+
|3000002    |Golden Crumbs   |{San Francisco, Mission, US}    |{37.7593, -122.4194}|
|3000000    |The Crumbly Nook|{Sydney, Bondi Beach, Australia}|{-33.8915, 151.2763}|
|3000001    |Tokyo Treats    |{Tokyo, Shibuya, Japan}         |{35.6586, 139.701}  |
+-----------+----------------+--------------------------------+--------------------+
only showing top 3 rows


Fields are read with a dotted path, or with `getField`. The special path
`location.*` expands every field of the struct into its own top-level
column, which is the usual way to flatten nested data:

In [10]:
stores.select("name", "location.city", F.col("coordinates").getField("latitude").alias("lat")).show(3)

+----------------+-------------+--------+
|            name|         city|     lat|
+----------------+-------------+--------+
|   Golden Crumbs|San Francisco| 37.7593|
|The Crumbly Nook|       Sydney|-33.8915|
|    Tokyo Treats|        Tokyo| 35.6586|
+----------------+-------------+--------+
only showing top 3 rows


In [11]:
stores.select("name", "location.*").show(3)

+----------------+-------------+-----------+---------+
|            name|         city|   district|  country|
+----------------+-------------+-----------+---------+
|   Golden Crumbs|San Francisco|    Mission|       US|
|The Crumbly Nook|       Sydney|Bondi Beach|Australia|
|    Tokyo Treats|        Tokyo|    Shibuya|    Japan|
+----------------+-------------+-----------+---------+
only showing top 3 rows


`withField` adds or replaces one field inside a struct, and `dropFields`
removes fields, without rebuilding the struct by hand:

In [12]:
stores.select(
    "name",
    F.col("location").withField("country", F.upper("location.country")).dropFields("district").alias("location"),
).show(3, truncate=False)

+----------------+-------------------+
|name            |location           |
+----------------+-------------------+
|Golden Crumbs   |{San Francisco, US}|
|The Crumbly Nook|{Sydney, AUSTRALIA}|
|Tokyo Treats    |{Tokyo, JAPAN}     |
+----------------+-------------------+
only showing top 3 rows


### Structs Inside Aggregations

Structs combine well with aggregation. `collect_list` on a struct keeps
related values together, and `sort_array` sorts structs by their first
field, then the second, and so on. That turns a customer's transactions into
a chronological purchase history:

In [13]:
history = transactions.groupBy("customerID").agg(
    F.sort_array(F.collect_list(F.struct("dateTime", "product", "quantity"))).alias("purchases")
)

first_customer = history.orderBy("customerID").first()
first_customer["purchases"][:3]

[Row(dateTime=datetime.datetime(2024, 5, 1, 5, 46, 20, 875401), product='Tokyo Tidbits', quantity=7),
 Row(dateTime=datetime.datetime(2024, 5, 2, 2, 47, 33, 593950), product='Austin Almond Biscotti', quantity=3),
 Row(dateTime=datetime.datetime(2024, 5, 2, 10, 56, 38, 464369), product='Tokyo Tidbits', quantity=36)]

In Python, each struct comes back as a `Row`, so `purchase.product` and
`purchase["product"]` both work.

## Maps

A map stores key-value pairs where the keys are not known in advance, for
example quantities per product or arbitrary tags. Here is each franchise's
items sold per product, as a map:

In [14]:
items_per_product = (
    transactions
    .groupBy("franchiseID", "product")
    .agg(F.sum("quantity").alias("items"))
    .groupBy("franchiseID")
    .agg(F.map_from_entries(F.collect_list(F.struct("product", "items"))).alias("items_by_product"))
)

items_per_product.printSchema()
items_per_product.orderBy("franchiseID").show(2, truncate=False)

root
 |-- franchiseID: long (nullable = true)
 |-- items_by_product: map (nullable = false)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)



+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|franchiseID|items_by_product                                                                                                                                 |
+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|3000000    |{Austin Almond Biscotti -> 142, Outback Oatmeal -> 190, Tokyo Tidbits -> 186, Golden Gate Ginger -> 184, Pearly Pies -> 176, Orchard Oasis -> 52}|
|3000001    |{Pearly Pies -> 74, Tokyo Tidbits -> 38, Austin Almond Biscotti -> 16, Orchard Oasis -> 53, Outback Oatmeal -> 72, Golden Gate Ginger -> 40}     |
+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------+
only showing top 2 rows


Look up a key with `[key]` or `element_at`, which return `NULL` for a key
that is not present. `map_keys`, `map_values` and `map_entries` return the
contents as arrays:

In [15]:
items_per_product.orderBy("franchiseID").select(
    "franchiseID",
    F.col("items_by_product")["Pearly Pies"].alias("pearly_pies"),
    F.size("items_by_product").alias("products_sold"),
    F.array_sort(F.map_keys("items_by_product")).alias("products"),
).show(3, truncate=False)

+-----------+-----------+-------------+--------------------------------------------------------------------------------------------------------+
|franchiseID|pearly_pies|products_sold|products                                                                                                |
+-----------+-----------+-------------+--------------------------------------------------------------------------------------------------------+
|3000000    |176        |6            |[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]|
|3000001    |74         |6            |[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]|
|3000002    |102        |6            |[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]|
+-----------+-----------+-------------+-------------------------------------------------------------------------------------------

Exploding a map produces two columns, `key` and `value`:

In [16]:
items_per_product.orderBy("franchiseID").limit(1).select("franchiseID", F.explode("items_by_product")).show(truncate=False)

+-----------+----------------------+-----+
|franchiseID|key                   |value|
+-----------+----------------------+-----+
|3000000    |Austin Almond Biscotti|142  |
|3000000    |Orchard Oasis         |52   |
|3000000    |Tokyo Tidbits         |186  |
|3000000    |Golden Gate Ginger    |184  |
|3000000    |Outback Oatmeal       |190  |
|3000000    |Pearly Pies           |176  |
+-----------+----------------------+-----+



:::{admonition} Note: Struct or Map?
:class: note

Use a **struct** when the set of fields is fixed and known, such as an
address. Every field can have its own type, and Spark can read single fields
efficiently. Use a **map** when the keys are data, such as product names or
user-defined tags, and all values share one type.
:::

## JSON Strings

Semi-structured data often arrives as JSON text. `from_json` parses a JSON
string into a struct, given a schema in DDL form, and `to_json` does the
reverse:

In [17]:
events = spark.createDataFrame(
    [
        ('{"franchise": "Tokyo Treats", "rating": 5, "tags": ["sweet", "fresh"]}',),
        ('{"franchise": "Golden Crumbs", "rating": 3, "tags": ["crowded"]}',),
        ('{"franchise": "Sugar Rush", "rating": "four"}',),
    ],
    schema="payload STRING",
)

parsed = events.select(
    F.from_json("payload", "franchise STRING, rating INT, tags ARRAY<STRING>").alias("event")
)

parsed.select("event.*").show(truncate=False)

+-------------+------+--------------+
|franchise    |rating|tags          |
+-------------+------+--------------+
|Tokyo Treats |5     |[sweet, fresh]|
|Golden Crumbs|3     |[crowded]     |
|Sugar Rush   |NULL  |NULL          |
+-------------+------+--------------+



Values that do not fit the schema become `NULL` rather than failing, as the
rating `"four"` shows. When you do not know the schema, `schema_of_json`
infers it from an example document:

In [18]:
spark.range(1).select(F.schema_of_json(F.lit(events.first()["payload"])).alias("schema")).show(truncate=False)

+--------------------------------------------------------------+
|schema                                                        |
+--------------------------------------------------------------+
|STRUCT<franchise: STRING, rating: BIGINT, tags: ARRAY<STRING>>|
+--------------------------------------------------------------+



Writing a struct back out as JSON:

In [19]:
stores.select("name", F.to_json("location").alias("location_json")).show(3, truncate=False)

+----------------+----------------------------------------------------------------+
|name            |location_json                                                   |
+----------------+----------------------------------------------------------------+
|Golden Crumbs   |{"city":"San Francisco","district":"Mission","country":"US"}    |
|The Crumbly Nook|{"city":"Sydney","district":"Bondi Beach","country":"Australia"}|
|Tokyo Treats    |{"city":"Tokyo","district":"Shibuya","country":"Japan"}         |
+----------------+----------------------------------------------------------------+
only showing top 3 rows


## The `VARIANT` Type

`from_json` needs a schema up front, and the schema has to be right for
every row. When documents vary in shape or evolve over time, Spark 4.0's
**`VARIANT`** type is the better fit. `parse_json` stores the parsed
document in an efficient binary format without any schema, and you extract
values with a path when you read them:

In [20]:
variants = events.select(F.parse_json("payload").alias("event"))

variants.printSchema()

root
 |-- event: variant (nullable = true)



In [21]:
variants.select(
    F.variant_get("event", "$.franchise", "string").alias("franchise"),
    F.try_variant_get("event", "$.rating", "int").alias("rating"),
    F.variant_get("event", "$.tags[0]", "string").alias("first_tag"),
).show()

+-------------+------+---------+
|    franchise|rating|first_tag|
+-------------+------+---------+
| Tokyo Treats|     5|    sweet|
|Golden Crumbs|     3|  crowded|
|   Sugar Rush|  NULL|     NULL|
+-------------+------+---------+



`variant_get` raises an error when a value cannot be converted to the
requested type, while `try_variant_get` returns `NULL`, which is why it is
used for the inconsistent `rating` field. A path that does not exist, such
as the missing `tags` of the third event, simply yields `NULL`.

In SQL, the shorter `:` path syntax does the same, with `::` for the cast:

In [22]:
variants.createOrReplaceTempView("review_events")

spark.sql("""
    SELECT event:franchise::string AS franchise,
           event:tags[0]::string   AS first_tag
    FROM review_events
""").show()

+-------------+---------+
|    franchise|first_tag|
+-------------+---------+
| Tokyo Treats|    sweet|
|Golden Crumbs|  crowded|
|   Sugar Rush|     NULL|
+-------------+---------+



:::{admonition} Note: When to Use `VARIANT`
:class: note

`VARIANT` is the recommended way on Databricks to store JSON whose shape is
not fixed, and it reads much faster than re-parsing JSON strings. When the
structure *is* fixed and well known, a struct with an explicit schema is
still the better choice, because it documents the data and lets Spark check
types up front.
:::

## Resources

- [Collection functions (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#collection-functions)
- [JSON functions (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#json-functions)
- [Query variant data (Databricks)](https://docs.databricks.com/aws/en/semi-structured/variant)
- [Higher-order functions (Databricks)](https://docs.databricks.com/aws/en/semi-structured/higher-order-functions)